# Data Analyst Agent — Reproducible Demo

This notebook walks through the agent end-to-end: 
1. Environment setup & imports
2. Single-turn tool-use trace
3. Multi-turn analysis with visualisations
4. Manual evaluation of agent outputs

> **Prerequisites**: set `GOOGLE_API_KEY` in your environment, then run `pip install -r ../requirements.txt`.

In [ ]:
import sys, os
sys.path.insert(0, '..')  # make src/ importable

# Verify API key is present
assert os.environ.get('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY first'
print('API key detected ✓')

## 1. Inspect the sample dataset

In [ ]:
import pandas as pd

df = pd.read_csv('../data/sales_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

## 2. Simple tool-call — load & describe

In [ ]:
from src.agent import run_agent, TextStep, ToolCallStep

steps_log = []

def collect(step):
    steps_log.append(step)
    if isinstance(step, ToolCallStep):
        print(f'TOOL: {step.tool_name}  keys={list(step.tool_input.keys())}')
    elif isinstance(step, TextStep):
        print(f'TEXT: {step.text[:200]}')

result = run_agent(
    'Load ../data/sales_data.csv as "sales" and give me a brief overview of the dataset.',
    on_step=collect
)

print('\n=== FINAL ANSWER ===')
print(result.final_answer)

## 3. Multi-step analysis — top products by profit margin

In [ ]:
steps_log.clear()

result2 = run_agent(
    'Using the sales dataset (already loaded as "sales"), '
    'compute profit margin (profit/revenue) per product, '
    'rank them, and create a bar chart.',
    on_step=collect
)

print('\n=== FINAL ANSWER ===')
print(result2.final_answer)
print(f'\nIterations used: {result2.iterations}')

In [ ]:
# Display any charts saved to outputs/
from pathlib import Path
from IPython.display import Image, display

for png in sorted(Path('../outputs').glob('*.png'))[-4:]:
    print(png.name)
    display(Image(str(png), width=800))

## 4. Correlation analysis

In [ ]:
result3 = run_agent(
    'Compute the correlation matrix for numeric columns in the "sales" dataset '
    'and save a heatmap.  Which variables are most strongly correlated?',
    on_step=collect
)
print(result3.final_answer)

## 5. Manual evaluation — step trace inspection

In [ ]:
tool_calls = [s for s in steps_log if isinstance(s, ToolCallStep)]
print(f'Total tool calls across last run: {len(tool_calls)}')
for tc in tool_calls:
    status = 'OK' if 'error' not in tc.tool_result else 'ERROR'
    print(f'  [{status}] {tc.tool_name}')

## 6. Regional revenue comparison

In [ ]:
result4 = run_agent(
    'Which region had the highest total revenue and profit? '
    'Create a grouped bar chart comparing revenue and profit by region.',
    on_step=collect
)
print(result4.final_answer)

## 7. Save a full analysis report

In [ ]:
result5 = run_agent(
    'Write a comprehensive 400-word executive summary of the sales dataset insights '
    'and save it to a file called "executive_summary".',
    on_step=collect
)
print(result5.final_answer)

---
*Generated by the Data Analyst Agent demo notebook.*